# HW4. Olist E-commerce in PostgreSQL: Typed Schema, DML, Segmentation and JOINs

**Author:** Serhii Kravchenko
**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, `olistbr/brazilian-ecommerce`), six of its nine CSV files
**Environment:** Google Colab (also runs locally), PostgreSQL 16 via `pgserver`, SQL cells via `jupysql`

The notebook loads the Olist CSV files into raw tables, moves them into a typed schema with keys and constraints, and from then on works only with the typed tables: DML with PostgreSQL features (`RETURNING`, `ON CONFLICT`, `UPDATE ... FROM`), a customer segmentation table, JOIN queries, aggregates, set operations and a window function. Every query is followed by an interpretation of its actual output.

Each section drops and recreates the tables it builds, so **Restart session and run all** always starts from a clean state and gives the same results. The only values that change between runs are the `created_at`, `updated_at` and `assigned_at` timestamps, which record when a row was written.

## 1. Setup and typed schema

### 1.1 Environment

`pgserver` ships PostgreSQL 16 as a Python package, so the database runs inside the notebook session. Its data directory is `hw4_pg` in the system temp folder, which is `/tmp/hw4_pg` on Colab.

A few environment details:
- Colab now defaults to Python 3.13, while `pgserver` 0.1.4 only publishes wheels up to Python 3.12. Its native module is built for the stable ABI (`abi3`), so the 3.12 wheel works on 3.13 once the wheel tag is renamed. The install cell handles this automatically.
- The session time zone is set to UTC, so the `TIMESTAMPTZ` audit columns (`created_at`, `updated_at`, `assigned_at`) look the same on every machine. The Windows build of `pgserver` comes without the time zone database and does not even know the name `UTC`; there the notebook uses the equivalent POSIX rule `UTC0` (zero offset, no daylight saving time). The saved outputs come from a local Windows run, so they show that name. The Olist timestamps themselves are local Brazilian time without an offset and are stored as `TIMESTAMP`, as the typed schema prescribes.
- `XDG_RUNTIME_DIR` is set only to silence a harmless `platformdirs` warning.

In [1]:
import glob, os, subprocess, sys, tempfile

os.environ.setdefault("XDG_RUNTIME_DIR", tempfile.mkdtemp())


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "kagglehub")

try:
    import pgserver
except ImportError:
    if sys.version_info >= (3, 13):
        tmp = tempfile.mkdtemp()
        subprocess.check_call([
            sys.executable, "-m", "pip", "download", "pgserver==0.1.4",
            "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
            "-d", tmp, "-q",
        ])
        wheel = glob.glob(os.path.join(tmp, "pgserver-*.whl"))[0]
        fixed = wheel.replace("-cp312-cp312-", "-cp312-abi3-")
        os.rename(wheel, fixed)
        pip_install(fixed)
    else:
        pip_install("pgserver==0.1.4")
    import pgserver

print("Python", sys.version.split()[0], "| pgserver ready")

Python 3.11.0 | pgserver ready


In [2]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import DBAPIError

PGDATA = Path(tempfile.gettempdir()) / "hw4_pg"
pg = pgserver.get_server(PGDATA, cleanup_mode="stop")
DB_URI = pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1)

probe = create_engine(DB_URI, future=True)
with probe.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())
    try:
        conn.execute(text("SET TIME ZONE 'UTC'"))
        SESSION_TZ = "UTC"
    except DBAPIError:
        SESSION_TZ = "UTC0"
probe.dispose()

engine = create_engine(DB_URI, future=True, connect_args={"options": f"-c timezone={SESSION_TZ}"})
print("session time zone:", SESSION_TZ)

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit
session time zone: UTC0


In [3]:
def q(query: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)


%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100
%sql engine

In [4]:
%%sql
SELECT current_setting('server_version') AS pg_version,
       current_setting('TimeZone')       AS session_tz;

pg_version,session_tz
16.2,UTC0


### 1.2 Getting the data

**Source:** the Kaggle dataset [`olistbr/brazilian-ecommerce`](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce): about 100,000 orders placed on the Olist marketplace in Brazil between 2016 and 2018, split into related CSV files (customers, orders, order items, products, sellers, reviews, payments, geolocation).

**Download.** `kagglehub` downloads public datasets without Kaggle credentials and caches them, so nothing has to be configured in Colab (variant A). If the CSV files are placed in `data/olist/` next to the notebook, for example from the course archive, the notebook uses that copy and downloads nothing (variant B).

Only six of the nine files are used. Payments, geolocation and the category name translation are not needed by any task.

In [5]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm without ipywidgets

import kagglehub

files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

LOCAL_DIR = Path.cwd() / "data" / "olist"
if all((LOCAL_DIR / file_name).exists() for file_name in files.values()):
    dataset_dir = LOCAL_DIR
    print("Source: local copy in data/olist/")
else:
    dataset_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    print(f"Source: Kaggle olistbr/brazilian-ecommerce, version {dataset_dir.name}, via kagglehub")

for file_name in files.values():
    print(f"  {file_name:<34} {(dataset_dir / file_name).stat().st_size / 1e6:5.1f} MB")

Source: Kaggle olistbr/brazilian-ecommerce, version 2, via kagglehub
  olist_customers_dataset.csv          9.0 MB
  olist_orders_dataset.csv            17.7 MB
  olist_order_items_dataset.csv       15.4 MB
  olist_products_dataset.csv           2.4 MB
  olist_sellers_dataset.csv            0.2 MB
  olist_order_reviews_dataset.csv     14.5 MB


### 1.3 Raw tables

The raw tables mirror the CSV files one to one and serve only as a landing layer. `pandas` infers the column types and `to_sql(if_exists="replace")` recreates the tables on every run. All later sections read from the typed tables only.

In [6]:
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)
olist_order_reviews_raw (99224, 7)


The types that `pandas` inferred show why every column is cast explicitly on the way to the typed tables. Timestamps arrive as text, and integer counts with gaps arrive as `double precision`, because `NaN` only exists for floats. Every column not listed here is `text`.

In [7]:
%%sql
SELECT table_name, column_name, data_type
FROM information_schema.columns
WHERE table_name LIKE 'olist%raw'
  AND data_type <> 'text'
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
olist_customers_raw,customer_zip_code_prefix,bigint
olist_order_items_raw,order_item_id,bigint
olist_order_items_raw,price,double precision
olist_order_items_raw,freight_value,double precision
olist_order_reviews_raw,review_score,bigint
olist_products_raw,product_name_lenght,double precision
olist_products_raw,product_description_lenght,double precision
olist_products_raw,product_photos_qty,double precision
olist_products_raw,product_weight_g,double precision
olist_products_raw,product_length_cm,double precision


### 1.4 Typed schema

The DDL below is used exactly as given in the assignment; the cell only adds a `SELECT` at the end that lists the created tables. Its main decisions:
- **Primary keys.** Olist's own hash-like IDs are the natural keys of customers, orders, products and sellers. An order item is identified by `(order_id, order_item_id)`. Reviews get a surrogate identity key `review_row_id`, because `review_id` is not unique in the source (see 1.7).
- **Foreign keys** follow the business relations: an order belongs to a customer, an item belongs to an order and points to a product and a seller, and a review belongs to an order.
- **CHECK constraints** encode the domain: the eight known order statuses, non-negative sizes and prices, and a review score from 1 to 5.
- **Indexes** support the lookups used later: `customer_unique_id` for the customer dimension in section 3, `(seller_id, order_id)` for the seller-level aggregations in sections 2 and 4, and `order_id` for joining reviews to orders.

In [8]:
%%sql
DROP TABLE IF EXISTS customer_segments    CASCADE;
DROP TABLE IF EXISTS seller_score         CASCADE;
DROP TABLE IF EXISTS seller_alerts        CASCADE;
DROP TABLE IF EXISTS olist_customer_dim   CASCADE;
DROP TABLE IF EXISTS olist_order_reviews  CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

CREATE TABLE olist_order_reviews (
    review_row_id            BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    review_id                TEXT,
    order_id                 TEXT NOT NULL REFERENCES olist_orders(order_id),
    review_score             SMALLINT CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title     TEXT,
    review_comment_message   TEXT,
    review_creation_date     TIMESTAMP,
    review_answer_timestamp  TIMESTAMP
);

CREATE INDEX idx_olist_order_reviews_order_id
    ON olist_order_reviews (order_id);

SELECT table_name, COUNT(*) AS n_columns
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name LIKE 'olist%'
  AND table_name NOT LIKE '%raw'
GROUP BY table_name
ORDER BY table_name;

table_name,n_columns
olist_customers,5
olist_order_items,7
olist_order_reviews,8
olist_orders,8
olist_products,9
olist_sellers,4


The constraints that were actually created, per table:

In [9]:
%%sql
SELECT
    c.conrelid::regclass::TEXT                       AS table_name,
    COUNT(*) FILTER (WHERE c.contype = 'p')          AS primary_keys,
    COUNT(*) FILTER (WHERE c.contype = 'f')          AS foreign_keys,
    COUNT(*) FILTER (WHERE c.contype = 'c')          AS checks,
    STRING_AGG(pg_get_constraintdef(c.oid), '; ' ORDER BY c.conname)
        FILTER (WHERE c.contype = 'f')               AS foreign_key_definitions
FROM pg_constraint AS c
WHERE c.conrelid::regclass::TEXT LIKE 'olist%'
GROUP BY c.conrelid
ORDER BY table_name;

table_name,primary_keys,foreign_keys,checks,foreign_key_definitions
olist_customers,1,0,0,None
olist_order_items,1,3,2,FOREIGN KEY (order_id) REFERENCES olist_orders(order_id); FOREIGN KEY (product_id) REFERENCES olist_products(product_id); FOREIGN KEY (seller_id) REFERENCES olist_sellers(seller_id)
olist_order_reviews,1,1,1,FOREIGN KEY (order_id) REFERENCES olist_orders(order_id)
olist_orders,1,1,1,FOREIGN KEY (customer_id) REFERENCES olist_customers(customer_id)
olist_products,1,0,7,None
olist_sellers,1,0,0,None


### 1.5 Moving the data into the typed tables

Each typed table is filled with `INSERT INTO ... SELECT` and an explicit cast per column. The order follows the foreign keys: customers before orders, orders, products and sellers before order items, and orders before reviews.

The source CSV spells two columns as `product_name_lenght` and `product_description_lenght`. The typed table uses the normalised names `product_name_length` and `product_description_length`, while the `SELECT` reads the raw columns under their original names.

In [10]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

INSERT INTO olist_order_reviews (
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
)
SELECT
    review_id,
    order_id,
    review_score::SMALLINT,
    review_comment_title,
    review_comment_message,
    review_creation_date::TIMESTAMP,
    review_answer_timestamp::TIMESTAMP
FROM olist_order_reviews_raw;

SELECT relname                                     AS table_name,
       pg_size_pretty(pg_total_relation_size(oid)) AS size_with_indexes
FROM pg_class
WHERE relkind = 'r'
  AND relname LIKE 'olist%'
  AND relname NOT LIKE '%raw'
ORDER BY pg_total_relation_size(oid) DESC;

table_name,size_with_indexes
olist_order_items,38 MB
olist_customers,26 MB
olist_order_reviews,25 MB
olist_orders,22 MB
olist_products,6072 kB
olist_sellers,552 kB


### 1.6 Smoke test of the row counts

In [11]:
%%sql
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
UNION ALL SELECT 'reviews', COUNT(*) FROM olist_order_reviews
ORDER BY table_name;

table_name,n_rows
customers,99441
order_items,112650
orders,99441
products,32951
reviews,99224
sellers,3095


The same counts next to the raw tables:

In [12]:
%%sql
SELECT table_name, raw_rows, typed_rows, raw_rows - typed_rows AS lost_rows
FROM (
    SELECT 'customers' AS table_name,
           (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
           (SELECT COUNT(*) FROM olist_customers)     AS typed_rows
    UNION ALL
    SELECT 'orders',
           (SELECT COUNT(*) FROM olist_orders_raw),
           (SELECT COUNT(*) FROM olist_orders)
    UNION ALL
    SELECT 'order_items',
           (SELECT COUNT(*) FROM olist_order_items_raw),
           (SELECT COUNT(*) FROM olist_order_items)
    UNION ALL
    SELECT 'products',
           (SELECT COUNT(*) FROM olist_products_raw),
           (SELECT COUNT(*) FROM olist_products)
    UNION ALL
    SELECT 'sellers',
           (SELECT COUNT(*) FROM olist_sellers_raw),
           (SELECT COUNT(*) FROM olist_sellers)
    UNION ALL
    SELECT 'reviews',
           (SELECT COUNT(*) FROM olist_order_reviews_raw),
           (SELECT COUNT(*) FROM olist_order_reviews)
) AS counts
ORDER BY table_name;

table_name,raw_rows,typed_rows,lost_rows
customers,99441,99441,0
order_items,112650,112650,0
orders,99441,99441,0
products,32951,32951,0
reviews,99224,99224,0
sellers,3095,3095,0


**Result:** every typed table has exactly as many rows as its raw table. No row was lost, so every row satisfied the keys, foreign keys and `CHECK` constraints, and there is no shortfall to explain. The check is still worth keeping: a typed table with fewer rows would mean that an `INSERT` failed halfway or that a cast or filter dropped data silently.

The reviews show why the schema needs the surrogate key `review_row_id`. The 99,224 review rows belong to 98,673 orders, and 789 `review_id` values occur in more than one row (1.7), so neither `order_id` nor `review_id` could serve as the primary key.

### 1.7 Data-quality profile

A short profile of the issues that matter for the later sections. The reflection in 6.4 refers back to it.

In [13]:
%%sql
SELECT check_name, n_rows
FROM (
    SELECT 1 AS ord, 'orders without any item' AS check_name,
           (SELECT COUNT(*) FROM olist_orders AS o
            WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS oi
                              WHERE oi.order_id = o.order_id)) AS n_rows
    UNION ALL
    SELECT 2, 'orders without any review',
           (SELECT COUNT(*) FROM olist_orders AS o
            WHERE NOT EXISTS (SELECT 1 FROM olist_order_reviews AS rv
                              WHERE rv.order_id = o.order_id))
    UNION ALL
    SELECT 3, 'orders with more than one review row',
           (SELECT COUNT(*) FROM (SELECT order_id FROM olist_order_reviews
                                  GROUP BY order_id HAVING COUNT(*) > 1) AS r)
    UNION ALL
    SELECT 4, 'review_id values shared by several rows',
           (SELECT COUNT(*) FROM (SELECT review_id FROM olist_order_reviews
                                  GROUP BY review_id HAVING COUNT(*) > 1) AS r)
    UNION ALL
    SELECT 5, 'reviews without a comment message',
           (SELECT COUNT(*) FROM olist_order_reviews
            WHERE NULLIF(TRIM(review_comment_message), '') IS NULL)
    UNION ALL
    SELECT 6, 'products without a category',
           (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL)
    UNION ALL
    SELECT 7, 'delivered orders without a delivery date',
           (SELECT COUNT(*) FROM olist_orders
            WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL)
    UNION ALL
    SELECT 8, 'not delivered orders with a delivery date',
           (SELECT COUNT(*) FROM olist_orders
            WHERE order_status <> 'delivered' AND order_delivered_customer_date IS NOT NULL)
    UNION ALL
    SELECT 9, 'customers (customer_id) per real buyer > 1',
           (SELECT COUNT(*) FROM (SELECT customer_unique_id FROM olist_customers
                                  GROUP BY customer_unique_id HAVING COUNT(*) > 1) AS r)
) AS dq
ORDER BY ord;

check_name,n_rows
orders without any item,775
orders without any review,768
orders with more than one review row,547
review_id values shared by several rows,789
reviews without a comment message,58256
products without a category,610
delivered orders without a delivery date,8
not delivered orders with a delivery date,6
customers (customer_id) per real buyer > 1,2997


In [14]:
%%sql
SELECT
    order_status,
    COUNT(*)                                                AS n_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)      AS pct,
    COUNT(*) FILTER (WHERE NOT EXISTS (
        SELECT 1 FROM olist_order_items AS oi WHERE oi.order_id = o.order_id
    ))                                                      AS orders_without_items
FROM olist_orders AS o
GROUP BY order_status
ORDER BY n_orders DESC;

order_status,n_orders,pct,orders_without_items
delivered,96478,97.02,0
shipped,1107,1.11,1
canceled,625,0.63,164
unavailable,609,0.61,603
invoiced,314,0.32,2
processing,301,0.30,0
created,5,0.01,5
approved,2,0.00,0


**Interpretation:** structurally the data is clean, since no row broke a constraint, but the content has gaps:
- **Orders without items.** 775 orders have no items, almost all of them `unavailable` (603) or `canceled` (164): the order was created, but nothing was shipped. Sums over items simply miss them, while counts over orders include them.
- **Reviews.** 768 orders have no review and 547 have more than one, and 789 `review_id` values are attached to several orders. A join from reviews to orders or items can therefore change the number of rows, which is why 2.2 and 4.2 aggregate reviews per order first.
- **Empty texts.** 58,256 of the 99,224 reviews (58.7%) have no comment, so text features would be missing for most orders.
- **Missing categories.** 610 products have no category; their items show up as a `NULL` category in 5.1.
- **Status vs. dates.** 8 delivered orders have no delivery date, and 6 orders with another status have one.
- **One buyer, several IDs.** 2,997 real buyers have more than one `customer_id`, which is the reason for the customer dimension in section 3.

97% of the orders are `delivered`. The 5 `created` and 2 `approved` orders look stuck, since they never moved on to a later status.

## 2. DML with PostgreSQL features

Four required statements (2.1–2.4) plus a few supporting ones. For each one the notebook explains what it changes, whether it is safe to run again, and what `RETURNING` adds.

### 2.1 `INSERT ... RETURNING`: an alert for the seller with the most late deliveries

**What it changes.** It creates `seller_alerts` and inserts one `late_delivery` alert with severity 4 for the seller with the most late delivered orders. An order is late when it reached the customer on a later calendar day than the estimated delivery date. `seller_orders` first reduces the items to distinct seller × order pairs, so an order with three items from the same seller counts as one late order, not three.

In [15]:
%%sql
DROP TABLE IF EXISTS seller_alerts CASCADE;

CREATE TABLE seller_alerts (
    alert_id   BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    seller_id  TEXT NOT NULL REFERENCES olist_sellers(seller_id),
    alert_type TEXT NOT NULL CHECK (alert_type IN ('late_delivery', 'low_rating', 'data_quality')),
    severity   SMALLINT NOT NULL CHECK (severity BETWEEN 1 AND 5),
    details    TEXT,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
late_sellers AS (
    SELECT
        so.seller_id,
        COUNT(DISTINCT o.order_id) AS late_orders
    FROM seller_orders AS so
    JOIN olist_orders AS o
        ON o.order_id = so.order_id
    WHERE o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    GROUP BY so.seller_id
    ORDER BY late_orders DESC, so.seller_id
    LIMIT 1
)
INSERT INTO seller_alerts (seller_id, alert_type, severity, details)
SELECT
    seller_id,
    'late_delivery',
    4,
    'Seller has ' || late_orders || ' late delivered orders in the loaded dataset'
FROM late_sellers
RETURNING alert_id, seller_id, alert_type, severity, created_at;

alert_id,seller_id,alert_type,severity,created_at
1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,2026-11-04 11:07:11.119236+00:00


**What `RETURNING` adds.** The statement hands back the rows exactly as they were stored, including the values the database generated itself: `alert_id` from the identity column and `created_at` from `DEFAULT NOW()`. There is no need for a second `SELECT` to find out which alert was created, and no risk of picking up a row that another session inserted in between. In the notebook this output is the audit log of what the cell changed; in a pipeline the same rows could be written to a log table or passed on to a notification step.

**Rerun.** The cell recreates `seller_alerts`, so running it again gives the same single alert. The `INSERT` on its own is not idempotent, though: running only the `INSERT` twice would create two identical alerts. A `UNIQUE (seller_id, alert_type)` constraint with `ON CONFLICT DO NOTHING` would make it safe to repeat.

To see whether the alert reflects a real problem or simply a large seller, the five sellers with the most late orders are shown next to their delivered volume:

In [16]:
%%sql
WITH seller_orders AS (
    SELECT DISTINCT oi.seller_id, oi.order_id
    FROM olist_order_items AS oi
)
SELECT
    so.seller_id,
    COUNT(*) FILTER (
        WHERE o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE
    )                                                        AS late_orders,
    COUNT(*) FILTER (WHERE o.order_delivered_customer_date IS NOT NULL)
                                                             AS delivered_orders,
    ROUND(100.0
          * COUNT(*) FILTER (
                WHERE o.order_delivered_customer_date::DATE > o.order_estimated_delivery_date::DATE)
          / NULLIF(COUNT(*) FILTER (WHERE o.order_delivered_customer_date IS NOT NULL), 0),
          1)                                                 AS late_pct
FROM seller_orders AS so
JOIN olist_orders AS o
    ON o.order_id = so.order_id
GROUP BY so.seller_id
ORDER BY late_orders DESC, so.seller_id
LIMIT 5;

seller_id,late_orders,delivered_orders,late_pct
4a3ca9315b744ce9f8e9374361493884,172,1772,9.7
1f50f920176fa81dab994f9023523100,124,1399,8.9
4869f7a5dfa277a7dca6462dcf3b52b2,118,1124,10.5
6560211a19b47992c3666cc44a7e94c0,96,1819,5.3
ea8482cd71df3c1969d7b9473ff13abc,96,1132,8.5


**Interpretation:** the alert went to seller `4a3ca931…` with 172 late orders. That seller also delivered 1,772 orders, so its late rate is 9.7%: above the 6.77% of all delivered orders (2.3), but not the worst in the top five, where seller `4869f7a5…` is late 10.5% of the time on fewer orders. An absolute count mostly finds the largest sellers. A production rule would combine a late rate with a minimum volume, for example a rate above 10% with at least 100 delivered orders.

### 2.2 `INSERT ... ON CONFLICT DO UPDATE`: an idempotent upsert of seller scores

**What it changes.** It creates `seller_score` and writes one row per seller that has sold anything: the average review score, the number of reviewed orders and a tier (gold from 4.5, silver from 3.5, bronze below). If a seller already has a row, the row is updated instead.

The next statements return thousands of rows each, so the notebook shows only the first ten of every result until the end of section 3.

In [17]:
%config SqlMagic.displaylimit = 10

In [18]:
%%sql
DROP TABLE IF EXISTS seller_score CASCADE;

CREATE TABLE seller_score (
    seller_id   TEXT PRIMARY KEY REFERENCES olist_sellers(seller_id),
    avg_rating  NUMERIC(3, 2),
    n_reviews   INTEGER NOT NULL DEFAULT 0 CHECK (n_reviews >= 0),
    tier        TEXT CHECK (tier IN ('gold', 'silver', 'bronze')),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
)
INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
SELECT
    seller_id,
    avg_rating,
    n_reviews,
    CASE
        WHEN avg_rating IS NULL THEN NULL
        WHEN avg_rating >= 4.5 THEN 'gold'
        WHEN avg_rating >= 3.5 THEN 'silver'
        ELSE 'bronze'
    END AS tier
FROM seller_rating
ON CONFLICT (seller_id) DO UPDATE
SET avg_rating = EXCLUDED.avg_rating,
    n_reviews  = EXCLUDED.n_reviews,
    tier       = EXCLUDED.tier,
    updated_at = NOW()
RETURNING seller_id, avg_rating, n_reviews, tier, updated_at;

seller_id,avg_rating,n_reviews,tier,updated_at
a61cc04793308395a840807104365121,5.00,1,gold,2026-11-04 11:07:11.583425+00:00
6d2f2e3b539480db1e0842b3a4e32e6e,3.67,3,silver,2026-11-04 11:07:11.583425+00:00
c53bcd3be457a342a97e39e5a9f0be22,3.20,5,bronze,2026-11-04 11:07:11.583425+00:00
9591fc341b1bfb7ef561e2968ec6e011,1.00,1,bronze,2026-11-04 11:07:11.583425+00:00
f9ec7093df3a7b346b7bcf7864069ca3,4.78,9,gold,2026-11-04 11:07:11.583425+00:00
fc38b5dceee1a730fad8853453437fbd,4.60,5,gold,2026-11-04 11:07:11.583425+00:00
5def4c3732941a971cba8fdee992ede1,4.60,5,gold,2026-11-04 11:07:11.583425+00:00
b39d7fe263ef469605dbb32608aee0af,3.88,43,silver,2026-11-04 11:07:11.583425+00:00
ea65d8b58316a6f2362f2a9e4b3e86ad,4.00,9,silver,2026-11-04 11:07:11.583425+00:00
a45765f8afb1e594b22bf9e974b46765,4.00,1,silver,2026-11-04 11:07:11.583425+00:00


**Why the query is built this way:**
- **`SELECT DISTINCT` in `seller_orders`.** `olist_order_items` has one row per item, so an order with three items from one seller appears three times. `DISTINCT` reduces it to one seller × order pair; otherwise that order's review would count three times for the seller.
- **Reviews are aggregated per `order_id` first.** Some orders have more than one review row (1.7). `review_per_order` turns them into exactly one score per order, so the join to `seller_orders` is one-to-one at the order level and nothing gets multiplied.
- **`ON CONFLICT (seller_id) DO UPDATE` makes the run repeatable.** `seller_id` is the primary key. On a second run every seller already has a row, so PostgreSQL updates it with the newly computed values (`EXCLUDED.*`) instead of failing on the duplicate key or adding a second row. The table always ends up with one current row per seller.
- **Sellers without reviews.** `LEFT JOIN` keeps sellers whose orders have no review. For them `AVG` over no values is `NULL`, `COUNT(rpo.order_id)` counts no rows and gives 0, and the `CASE` maps a `NULL` rating to a `NULL` tier. "No reviews yet" stays distinguishable from "bad reviews".

**Dataset limitation.** In Olist a review belongs to an order, not to a seller. When one order contains items from several sellers, its review is credited to each of them, although the customer may have been unhappy with only one. This is an acceptable simplification for the exercise, but the score of a seller who often shares orders with others is less reliable.

The tiers that came out:

In [19]:
%%sql
SELECT
    COALESCE(tier, '(no reviews)')  AS tier,
    COUNT(*)                        AS n_sellers,
    ROUND(AVG(avg_rating), 2)       AS mean_rating,
    SUM(n_reviews)                  AS reviewed_orders
FROM seller_score
GROUP BY tier
ORDER BY MIN(avg_rating) DESC NULLS LAST;

tier,n_sellers,mean_rating,reviewed_orders
gold,983,4.83,7093
silver,1568,4.06,87123
bronze,539,2.30,5031
(no reviews),5,None,0


How much the naive item-level join from the assignment would distort the result, compared with the table above:

In [20]:
%%sql
WITH naive AS (
    SELECT
        oi.seller_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rv.review_score)              AS n_reviews
    FROM olist_order_items AS oi
    LEFT JOIN olist_order_reviews AS rv
        ON rv.order_id = oi.order_id
    GROUP BY oi.seller_id
)
SELECT
    SUM(n.n_reviews)                                              AS naive_review_count,
    SUM(s.n_reviews)                                              AS correct_review_count,
    COUNT(*) FILTER (WHERE n.avg_rating IS DISTINCT FROM s.avg_rating)
                                                                  AS sellers_with_other_rating,
    COUNT(*) FILTER (WHERE n.n_reviews <> s.n_reviews)            AS sellers_with_other_count,
    MAX(ABS(n.avg_rating - s.avg_rating))                         AS max_rating_shift
FROM naive AS n
JOIN seller_score AS s
    USING (seller_id);

naive_review_count,correct_review_count,sellers_with_other_rating,sellers_with_other_count,max_rating_shift
112372,99247,1264,1458,1.67


**Interpretation:** 983 sellers are gold, 1,568 silver and 539 bronze; 5 sellers have no reviewed order and keep a `NULL` rating and tier. Silver sellers carry most of the volume (87,123 reviewed orders). Gold is dominated by small sellers: the median gold seller has only 3 reviewed orders, and 634 of the 983 have fewer than five. A single happy customer is enough for a 5.00 rating, so a real ranking would need a minimum number of reviews or a rating shrunk towards the overall mean.

The naive item-level join counts 112,372 reviews instead of 99,247 (13% more) and gives 1,264 of the 3,095 sellers a different rating, by up to 1.67 points. The correct count of 99,247 is itself 1,330 higher than the 97,917 reviewed orders that have items: 1,278 orders contain items from more than one seller, and their review is credited to each of them. That is the dataset limitation described above, now in numbers.

**Second run.** The same upsert once more, this time wrapped in a data-modifying CTE that reports what happened instead of listing the rows. All parts of one `WITH` statement see the same snapshot, so `rows_before` is the table as it was before this upsert.

In [21]:
%%sql
WITH seller_orders AS (
    SELECT DISTINCT
        oi.seller_id,
        oi.order_id
    FROM olist_order_items AS oi
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
seller_rating AS (
    SELECT
        so.seller_id,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_rating,
        COUNT(rpo.order_id) AS n_reviews
    FROM seller_orders AS so
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = so.order_id
    GROUP BY so.seller_id
),
upserted AS (
    INSERT INTO seller_score (seller_id, avg_rating, n_reviews, tier)
    SELECT
        seller_id,
        avg_rating,
        n_reviews,
        CASE
            WHEN avg_rating IS NULL THEN NULL
            WHEN avg_rating >= 4.5 THEN 'gold'
            WHEN avg_rating >= 3.5 THEN 'silver'
            ELSE 'bronze'
        END AS tier
    FROM seller_rating
    ON CONFLICT (seller_id) DO UPDATE
    SET avg_rating = EXCLUDED.avg_rating,
        n_reviews  = EXCLUDED.n_reviews,
        tier       = EXCLUDED.tier,
        updated_at = NOW()
    RETURNING seller_id
)
SELECT
    (SELECT COUNT(*) FROM seller_score) AS rows_before,
    (SELECT COUNT(*) FROM upserted)     AS rows_returned;

rows_before,rows_returned
3095,3095


In [22]:
%%sql
SELECT COUNT(*) AS rows_after, COUNT(DISTINCT seller_id) AS distinct_sellers
FROM seller_score;

rows_after,distinct_sellers
3095,3095


**Interpretation:** the second run returned 3,095 rows, one per seller. The table had 3,095 rows before and still has 3,095 after, so every returned row was an update of an existing row and none was inserted. This is the idempotency `ON CONFLICT` provides: the statement can run after every data load and always leaves exactly one current score per seller.

### 2.3 `UPDATE ... FROM`: the late-delivery flag in `olist_orders`

**What it changes.** It adds the column `is_late BOOLEAN NOT NULL DEFAULT FALSE` (only if it does not exist yet) and sets it for every order from the delivery dates, using the same rule as in 2.1. `UPDATE ... FROM` joins the table to the `delivery_flags` CTE, and `RETURNING` lists the updated orders.

In [23]:
%%sql
ALTER TABLE olist_orders
    ADD COLUMN IF NOT EXISTS is_late BOOLEAN NOT NULL DEFAULT FALSE;

WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
RETURNING o.order_id, o.customer_id, o.is_late;

order_id,customer_id,is_late
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,False
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,False
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,False
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,False
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,False
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,False
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,False
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,False
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,False
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,False


**Rerun.** The statement is idempotent: it does not append `TRUE` to some rows, it recomputes the flag from the current delivery dates and assigns it to every order. A second run leaves the same values. `ADD COLUMN IF NOT EXISTS` makes the `ALTER TABLE` repeatable as well. The `IS NOT NULL` checks matter because a comparison with `NULL` gives `NULL`, which a `NOT NULL` column would reject; with them, an order that has not been delivered gets `FALSE`.

The flag per order status:

In [24]:
%%sql
SELECT
    order_status,
    COUNT(*)                                                    AS n_orders,
    COUNT(*) FILTER (WHERE is_late)                             AS n_late,
    ROUND(100.0 * COUNT(*) FILTER (WHERE is_late) / COUNT(*), 2) AS late_pct
FROM olist_orders
GROUP BY order_status
ORDER BY n_orders DESC;

order_status,n_orders,n_late,late_pct
delivered,96478,6534,6.77
shipped,1107,0,0.00
canceled,625,1,0.16
unavailable,609,0,0.00
invoiced,314,0,0.00
processing,301,0,0.00
created,5,0,0.00
approved,2,0,0.00


**Interpretation:** 6,534 of the 96,478 delivered orders (6.77%) arrived after the estimated date. The other statuses have no late orders, as expected, because they have no delivery date. The one exception is a `canceled` order that has a delivery date (2018-03-21, estimate 2018-03-09) and is therefore flagged as late. That is the status–date contradiction from 1.7; as an ML feature, `is_late` should probably be restricted to delivered orders.

The statement above rewrites every order on every run, even when the flag does not change. In PostgreSQL each updated row is a new row version, so a full rewrite creates dead tuples for `VACUUM` and fires any update triggers. A production version adds one condition and only touches rows whose flag actually changes. On this second run it returns nothing, which confirms that the first run already left the table in its final state:

In [25]:
%%sql
WITH delivery_flags AS (
    SELECT
        order_id,
        (
            order_delivered_customer_date IS NOT NULL
            AND order_estimated_delivery_date IS NOT NULL
            AND order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE
        ) AS is_late_calc
    FROM olist_orders
)
UPDATE olist_orders AS o
SET is_late = f.is_late_calc
FROM delivery_flags AS f
WHERE f.order_id = o.order_id
  AND o.is_late IS DISTINCT FROM f.is_late_calc
RETURNING o.order_id, o.is_late;

order_id,is_late


### 2.4 `DELETE ... RETURNING`: removing outdated alerts

**What it changes.** It deletes every alert older than 30 days and returns the deleted rows, so the output is a record of what was removed.

In [26]:
%%sql
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;

alert_id,seller_id,alert_type,created_at


**Why the result is empty.** `seller_alerts` is recreated in 2.1 on every run, so its only alert was created a few seconds ago and is not older than 30 days. An empty result is the correct outcome here, and the statement is safe to repeat: a second run finds nothing more to delete.

To check that the filter really works, an alert can be backdated. `created_at` only has a `DEFAULT`, so an explicit value is allowed, unlike `alert_id`, which is `GENERATED ALWAYS`. The extra alert below is a real `data_quality` finding, the seller with the most order items whose product has no category, recorded as if an earlier pipeline run had raised it 45 days ago:

In [27]:
%%sql
INSERT INTO seller_alerts (seller_id, alert_type, severity, details, created_at)
SELECT
    oi.seller_id,
    'data_quality',
    2,
    COUNT(*) || ' order items point to products without a category',
    NOW() - INTERVAL '45 days'
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
WHERE p.product_category_name IS NULL
GROUP BY oi.seller_id
ORDER BY COUNT(*) DESC, oi.seller_id
LIMIT 1
RETURNING alert_id, seller_id, alert_type, severity, details, created_at;

alert_id,seller_id,alert_type,severity,details,created_at
2,c826c40d7b19f62a09e2d7c5e7295ee2,data_quality,2,372 order items point to products without a category,2026-09-20 11:07:14.757312+00:00


The same `DELETE` again:

In [28]:
%%sql
DELETE FROM seller_alerts
WHERE created_at < NOW() - INTERVAL '30 days'
RETURNING alert_id, seller_id, alert_type, created_at;

alert_id,seller_id,alert_type,created_at
2,c826c40d7b19f62a09e2d7c5e7295ee2,data_quality,2026-09-20 11:07:14.757312+00:00


In [29]:
%%sql
SELECT alert_id, seller_id, alert_type, severity, details, created_at
FROM seller_alerts
ORDER BY alert_id;

alert_id,seller_id,alert_type,severity,details,created_at
1,4a3ca9315b744ce9f8e9374361493884,late_delivery,4,Seller has 172 late delivered orders in the loaded dataset,2026-11-04 11:07:11.119236+00:00


**Interpretation:** the second `DELETE` removed exactly the backdated `data_quality` alert (372 items of seller `c826c40d…` point to products without a category) and kept the fresh `late_delivery` alert. Because `RETURNING` lists what was deleted, the cell output doubles as a deletion log; in production these rows could be copied to an archive table in the same statement, through a data-modifying CTE.

## 3. `customer_segments`

Segmentation is computed per real buyer, `customer_unique_id`. In Olist a new `customer_id` is issued for every order, so per `customer_id` almost every "customer" would have exactly one order.

### 3.1 Customer dimension

`olist_customer_dim` holds one row per `customer_unique_id`, with the `customer_id`, state and city of that buyer's most recent order. `DISTINCT ON` keeps the first row of each buyer in the `ORDER BY` order: newest purchase first, `NULLS LAST` for a `customer_id` without orders, and `customer_id` as the tie-breaker.

In [30]:
%%sql
DROP TABLE IF EXISTS customer_segments CASCADE;
DROP TABLE IF EXISTS olist_customer_dim CASCADE;

CREATE TABLE olist_customer_dim (
    customer_unique_id TEXT PRIMARY KEY,
    latest_customer_id TEXT REFERENCES olist_customers(customer_id),
    customer_state     CHAR(2),
    customer_city      TEXT
);

INSERT INTO olist_customer_dim (
    customer_unique_id,
    latest_customer_id,
    customer_state,
    customer_city
)
SELECT DISTINCT ON (c.customer_unique_id)
    c.customer_unique_id,
    c.customer_id AS latest_customer_id,
    c.customer_state,
    c.customer_city
FROM olist_customers AS c
LEFT JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp DESC NULLS LAST,
    c.customer_id;

SELECT
    (SELECT COUNT(*) FROM olist_customers)                             AS customer_ids,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers)   AS unique_buyers,
    (SELECT COUNT(*) FROM olist_customer_dim)                          AS dim_rows;

customer_ids,unique_buyers,dim_rows
99441,96096,96096


### 3.2 DDL and idempotent fill

`customer_segments` uses an identity primary key, a `UNIQUE` business key, a foreign key to the dimension with `ON DELETE CASCADE`, `CHECK` constraints on every measure, `NOT NULL` and `DEFAULT` values.

In [31]:
%%sql
CREATE TABLE customer_segments (
    segment_id          BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,

    customer_unique_id  TEXT NOT NULL,
    segment_name        TEXT NOT NULL
        CHECK (segment_name IN ('VIP', 'regular', 'new', 'inactive')),

    rfm_score           SMALLINT NOT NULL
        CHECK (rfm_score BETWEEN 1 AND 5),

    recency_days        INTEGER
        CHECK (recency_days IS NULL OR recency_days >= 0),

    monetary_value      NUMERIC(12, 2) NOT NULL DEFAULT 0
        CHECK (monetary_value >= 0),

    n_orders            INTEGER NOT NULL DEFAULT 0
        CHECK (n_orders >= 0),

    assigned_at         TIMESTAMPTZ NOT NULL DEFAULT NOW(),

    UNIQUE (customer_unique_id)
);

ALTER TABLE customer_segments
    ADD CONSTRAINT fk_customer_segments_customer_unique
    FOREIGN KEY (customer_unique_id)
    REFERENCES olist_customer_dim(customer_unique_id)
    ON DELETE CASCADE;

SELECT column_name, data_type, is_nullable, column_default, is_identity
FROM information_schema.columns
WHERE table_name = 'customer_segments'
ORDER BY ordinal_position;

column_name,data_type,is_nullable,column_default,is_identity
segment_id,bigint,NO,None,YES
customer_unique_id,text,NO,None,NO
segment_name,text,NO,None,NO
rfm_score,smallint,NO,None,NO
recency_days,integer,YES,None,NO
monetary_value,numeric,NO,0,NO
n_orders,integer,NO,0,NO
assigned_at,timestamp with time zone,NO,now(),NO


The fill uses a `reference_date` taken from the data itself: the day after the last purchase in Olist.

In [32]:
%%sql
WITH reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
order_totals AS (
    SELECT
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(oi.price + oi.freight_value), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        o.customer_id,
        o.order_purchase_timestamp
),
customer_stats AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS monetary_value,
        MAX(ot.order_date) AS last_order_date
    FROM olist_customer_dim AS cd
    JOIN olist_customers AS c
        ON c.customer_unique_id = cd.customer_unique_id
    LEFT JOIN order_totals AS ot
        ON ot.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
),
segmented AS (
    SELECT
        cs.customer_unique_id,
        cs.n_orders,
        cs.monetary_value,
        CASE
            WHEN cs.last_order_date IS NULL THEN NULL
            ELSE (rd.as_of_date - cs.last_order_date)::INTEGER
        END AS recency_days,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 'VIP'
            WHEN cs.last_order_date IS NOT NULL
                 AND (rd.as_of_date - cs.last_order_date)::INTEGER <= 90
                 AND cs.n_orders = 1 THEN 'new'
            WHEN cs.n_orders >= 2 OR cs.monetary_value >= 100 THEN 'regular'
            ELSE 'inactive'
        END AS segment_name,
        CASE
            WHEN cs.monetary_value >= 1000 THEN 5
            WHEN cs.monetary_value >= 500  THEN 4
            WHEN cs.n_orders >= 2          THEN 3
            WHEN cs.n_orders = 1           THEN 2
            ELSE 1
        END::SMALLINT AS rfm_score
    FROM customer_stats AS cs
    CROSS JOIN reference_date AS rd
)
INSERT INTO customer_segments (
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
)
SELECT
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders
FROM segmented
ON CONFLICT (customer_unique_id) DO UPDATE
SET segment_name   = EXCLUDED.segment_name,
    rfm_score      = EXCLUDED.rfm_score,
    recency_days   = EXCLUDED.recency_days,
    monetary_value = EXCLUDED.monetary_value,
    n_orders       = EXCLUDED.n_orders,
    assigned_at    = NOW()
RETURNING
    customer_unique_id,
    segment_name,
    rfm_score,
    recency_days,
    monetary_value,
    n_orders;

customer_unique_id,segment_name,rfm_score,recency_days,monetary_value,n_orders
0000366f3b9a7992bf8c76cfdf3221e2,regular,2,161,141.90,1
0000b849f77a49e4a4ce2b2a4ca5be3f,inactive,2,164,27.19,1
0000f46a3911fa3c0805444483337064,inactive,2,587,86.22,1
0000f6ccb0745a6a4b88665a16c9f078,inactive,2,371,43.62,1
0004aac84e0df4da2b147fca70cf8255,regular,2,338,196.89,1
0004bd2a26a76fe21f786e4fbd80607f,regular,2,196,166.98,1
00050ab1314c0e55a6ca13cf7181fecf,inactive,2,181,35.38,1
00053a61a98854899e70ed204dd4bafe,regular,2,232,419.18,1
0005e1862207bf6ccc02e4228effd9a0,regular,2,593,150.12,1
0005ef4cd20d2893f0d9fbd94d3c0d97,regular,2,220,129.76,1


In [33]:
%config SqlMagic.displaylimit = 100

### 3.3 Constraint check

In [34]:
%%sql
SELECT
    conname,
    contype,
    pg_get_constraintdef(oid) AS definition
FROM pg_constraint
WHERE conrelid = 'customer_segments'::regclass
ORDER BY conname;

conname,contype,definition
customer_segments_customer_unique_id_key,u,UNIQUE (customer_unique_id)
customer_segments_monetary_value_check,c,CHECK ((monetary_value >= (0)::numeric))
customer_segments_n_orders_check,c,CHECK ((n_orders >= 0))
customer_segments_pkey,p,PRIMARY KEY (segment_id)
customer_segments_recency_days_check,c,CHECK (((recency_days IS NULL) OR (recency_days >= 0)))
customer_segments_rfm_score_check,c,CHECK (((rfm_score >= 1) AND (rfm_score <= 5)))
customer_segments_segment_name_check,c,"CHECK ((segment_name = ANY (ARRAY['VIP'::text, 'regular'::text, 'new'::text, 'inactive'::text])))"
fk_customer_segments_customer_unique,f,FOREIGN KEY (customer_unique_id) REFERENCES olist_customer_dim(customer_unique_id) ON DELETE CASCADE


`contype`: `p` primary key, `u` unique, `f` foreign key, `c` check. The `NOT NULL` rules are column properties in PostgreSQL 16 and are not listed in `pg_constraint`.

**Why the design looks like this:**
- **`customer_unique_id` instead of `customer_id`.** A `customer_id` exists once per order, so it cannot show repeat purchases: with it, frequency would be 1 for almost everyone and recency and monetary value would describe single orders, not buyers. `customer_unique_id` groups all orders of one real buyer.
- **Why `olist_customer_dim` is needed for the FK.** A foreign key must point to a column with a primary key or unique constraint. In `olist_customers`, `customer_unique_id` repeats for buyers with several orders, so it cannot be referenced. The dimension holds exactly one row per buyer and gives the FK a valid target. Thanks to `ON DELETE CASCADE`, removing a buyer from the dimension also removes their segment.
- **`reference_date` from the data, not `CURRENT_DATE`.** The last Olist order is from 2018. Measured from `CURRENT_DATE`, every buyer would have a recency of several years and nobody would ever count as `new`. Recency would also change every day, so the same notebook would give different segments on different days. A date derived from the data makes the result reproducible and puts "today" where the dataset ends.
- **How `ON CONFLICT` makes the segmentation repeatable.** `UNIQUE (customer_unique_id)` is the conflict target. A second run finds every buyer already present, so it updates the segment, score, measures and `assigned_at` in place instead of failing or adding a duplicate. `segment_id` stays the same, so anything that refers to it keeps working.

The reference date and the resulting segments:

In [35]:
%%sql
SELECT MAX(order_purchase_timestamp)          AS last_purchase,
       MAX(order_purchase_timestamp)::DATE + 1 AS as_of_date
FROM olist_orders;

last_purchase,as_of_date
2018-10-17 17:30:18,2018-10-18


In [36]:
%%sql
SELECT
    segment_name,
    rfm_score,
    COUNT(*)                                            AS n_buyers,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2)  AS pct,
    ROUND(AVG(n_orders), 2)                             AS avg_orders,
    ROUND(AVG(monetary_value), 2)                       AS avg_monetary,
    ROUND(AVG(recency_days))                            AS avg_recency_days
FROM customer_segments
GROUP BY segment_name, rfm_score
ORDER BY rfm_score DESC, segment_name;

segment_name,rfm_score,n_buyers,pct,avg_orders,avg_monetary,avg_recency_days
VIP,5,1191,1.24,1.11,1591.81,290
new,4,269,0.28,1.00,675.09,74
regular,4,2964,3.08,1.14,680.51,305
regular,3,2608,2.71,2.07,220.74,271
inactive,2,40032,41.66,1.00,59.28,316
new,2,8622,8.97,1.00,123.71,72
regular,2,40410,42.05,1.00,191.38,310


**Interpretation:**
- **Reference date.** The last purchase is from 2018-10-17, so `as_of_date` is 2018-10-18. Regular trading actually ends on 2018-08-31: the last 20 orders, from September and October, are all `canceled` or `shipped`. Those few stray orders move "today" 47 days forward, so every recency is 47 days longer than it would be relative to the end of trading, and the 90-day window for `new` starts on 2018-07-20. A more robust choice would be the last day with normal order volume, or a high percentile instead of `MAX`.
- **Segments.** `regular` (47.84%) and `inactive` (41.66%) cover almost everybody: 40,032 buyers placed a single order under R$100 more than 90 days before the reference date. `new` buyers make up 9.25%, and `VIP` buyers (R$1,000 or more) only 1.24%, with an average spend of R$1,591.81.
- **Frequency barely matters.** Only 2,997 of the 96,096 buyers (3.1%) ever ordered more than once. The average number of orders stays between 1.00 and 1.14 in every group except the `regular` buyers with score 3 (2.07). Olist is mostly a marketplace of one-off purchases.
- **The score is not a real RFM score.** It ranks by monetary value first and frequency second and ignores recency, so a VIP buyer from two years ago keeps score 5. For a churn or reactivation use case, recency would have to enter the score, for example as quintiles of recency, frequency and monetary value.

## 4. JOIN queries

The queries follow the assignment's templates. Where a `LIMIT` could cut between rows with equal values, the `ORDER BY` gets an extra key, so the output is the same on every run.

### 4.1 `INNER JOIN` on four tables: top categories and seller states by revenue

**Business-question:** Which product categories and seller states bring the most revenue in delivered orders?

Revenue here is item price plus freight, as paid by the customer. `INNER JOIN` keeps only items whose order, product and seller all exist, which is the case for every item thanks to the foreign keys.

In [37]:
%%sql
SELECT
    p.product_category_name,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS revenue
FROM olist_order_items AS oi
JOIN olist_orders AS o
    ON o.order_id = oi.order_id
JOIN olist_products AS p
    ON p.product_id = oi.product_id
JOIN olist_sellers AS s
    ON s.seller_id = oi.seller_id
WHERE o.order_status = 'delivered'
GROUP BY p.product_category_name, s.seller_state
ORDER BY revenue DESC NULLS LAST, p.product_category_name, s.seller_state
LIMIT 10;

product_category_name,seller_state,n_orders,revenue
cama_mesa_banho,SP,8214,1070048.14
relogios_presentes,SP,4494,1016168.80
beleza_saude,SP,5684,790967.66
esporte_lazer,SP,4772,680165.54
moveis_decoracao,SP,4597,605650.63
cool_stuff,SP,2621,484196.53
utilidades_domesticas,SP,4021,471810.66
automotivo,SP,3018,438833.22
informatica_acessorios,SP,2783,388506.42
bebes,SP,1981,339129.74


**Interpretation:** sellers from São Paulo (SP) take all ten places. The top pair is bed, bath and table (`cama_mesa_banho`) with R$1.07 million from 8,214 orders, followed by watches and gifts (`relogios_presentes`) with almost the same revenue from only 4,494 orders, about 1.7 times as much per order. Health and beauty, sports and furniture follow. The ranking mostly reflects where the sellers are (60% of them are in SP, see 4.3), so a comparison of categories across states should be normalised per seller or per order.

### 4.2 `LEFT JOIN` + `COALESCE`: customer-level features

**Business-question:** Which customer-level features can be built for a real buyer (`customer_unique_id`), and who are the biggest buyers?

`review_per_order` collapses the reviews to one score per order before the join. Without it, an order with several review rows would multiply its `order_total` in `total_spend` and give extra weight to its score. The `LEFT JOIN`s keep buyers and orders without items or reviews, and `COALESCE` turns their gaps into explicit defaults.

In [38]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_unique_id,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_unique_id
),
review_per_order AS (
    SELECT
        rv.order_id,
        AVG(rv.review_score)::NUMERIC(3, 2) AS order_review_score
    FROM olist_order_reviews AS rv
    WHERE rv.review_score IS NOT NULL
    GROUP BY rv.order_id
),
customer_stats AS (
    SELECT
        ot.customer_unique_id,
        COUNT(DISTINCT ot.order_id) AS n_orders,
        COALESCE(SUM(ot.order_total), 0)::NUMERIC(12, 2) AS total_spend,
        AVG(rpo.order_review_score)::NUMERIC(3, 2) AS avg_review_score
    FROM order_totals AS ot
    LEFT JOIN review_per_order AS rpo
        ON rpo.order_id = ot.order_id
    GROUP BY ot.customer_unique_id
)
SELECT
    cd.customer_unique_id,
    cd.customer_state,
    COALESCE(cs.n_orders, 0) AS n_orders,
    COALESCE(cs.total_spend, 0)::NUMERIC(12, 2) AS total_spend,
    COALESCE(cs.avg_review_score, 0)::NUMERIC(3, 2) AS avg_review_score
FROM olist_customer_dim AS cd
LEFT JOIN customer_stats AS cs
    ON cs.customer_unique_id = cd.customer_unique_id
ORDER BY total_spend DESC NULLS LAST, cd.customer_unique_id
LIMIT 20;

customer_unique_id,customer_state,n_orders,total_spend,avg_review_score
0a0a92112bd4c708ca5fde585afaa872,RJ,1,13664.08,1.00
da122df9eeddfedc1dc1f5349a1a690c,RJ,2,7571.63,5.00
763c8b1c9c68a0229c42c9fc6f662b93,ES,1,7274.88,1.00
dc4802a71eae9be1dd28f5d788ceb526,MS,1,6929.31,5.00
459bef486812aa25204be022145caa62,ES,1,6922.21,0.00
ff4159b92c40ebe40454e3e6a7c35ed6,SP,1,6726.66,5.00
4007669dec559734d6f53e029e360987,MG,1,6081.54,1.00
5d0a2980b292d049061542014e8960bf,GO,1,4809.44,1.00
eebb5dda148d3893cdaf5b5ca3040ccb,SP,1,4764.34,4.00
48e1ac109decbb87765a3eade6854098,PB,1,4681.78,5.00


**Interpretation:** the top buyers are mostly one-off buyers of expensive items: 17 of the 20 placed a single order. The largest, `0a0a9211…` from RJ, spent R$13,664.08 on one order of eight fixed-line phones (`telefonia_fixa`) and rated it 1, so a high spend says nothing about satisfaction.

The `COALESCE` defaults need care. For `n_orders` and `total_spend`, 0 really means "nothing". For `avg_review_score`, 0 only means "no review" (buyer `459bef48…`, one of 716 such buyers), and it lies outside the 1–5 scale, so a model would read it as an extremely unhappy customer. For ML, `NULL` plus a `has_review` flag, or the global mean, would be a safer default.

### 4.3 `FULL OUTER JOIN`: states of customers and sellers

**Business-question:** Which states are represented only among customers, only among sellers, or in both groups?

Customers are counted per `customer_id`, that is per order, as in the assignment.

In [39]:
%%sql
WITH customer_states AS (
    SELECT customer_state AS state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state
), seller_states AS (
    SELECT seller_state AS state, COUNT(*) AS n_sellers
    FROM olist_sellers
    GROUP BY seller_state
)
SELECT
    COALESCE(c.state, s.state) AS state,
    c.n_customers,
    s.n_sellers,
    CASE
        WHEN c.state IS NOT NULL AND s.state IS NOT NULL THEN 'both'
        WHEN c.state IS NOT NULL THEN 'customers_only'
        ELSE 'sellers_only'
    END AS state_presence
FROM customer_states AS c
FULL OUTER JOIN seller_states AS s
    ON s.state = c.state
ORDER BY state;

state,n_customers,n_sellers,state_presence
AC,81,1,both
AL,413,None,customers_only
AM,148,1,both
AP,68,None,customers_only
BA,3380,19,both
CE,1336,13,both
DF,2140,30,both
ES,2033,23,both
GO,2020,40,both
MA,747,1,both


**Interpretation:** customers come from all 27 Brazilian federal units, sellers from 23. Four states have customers but no seller: Alagoas (AL, 413 customers), Tocantins (TO, 280), Amapá (AP, 68) and Roraima (RR, 46). No state has sellers only. Sellers are far more concentrated than customers: SP alone has 1,849 of the 3,095 sellers (60%) but 41,746 of the 99,441 customers (42%), and five states have exactly one seller. Buyers in the customers-only states always order from another state, which means longer deliveries and more freight (see 5.2).

### 4.4 `SELF JOIN`: pairs of sellers from one state but different cities

**Business-question:** Which seller pairs could be compared regionally within one state?

`s1.seller_id < s2.seller_id` keeps each pair once and excludes pairing a seller with itself. `IS DISTINCT FROM` also treats a missing city as different from a known one.

In [40]:
%%sql
SELECT
    s1.seller_id AS seller_a,
    s2.seller_id AS seller_b,
    s1.seller_state,
    s1.seller_city AS city_a,
    s2.seller_city AS city_b
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
ORDER BY s1.seller_state, seller_a, seller_b
LIMIT 20;

seller_a,seller_b,seller_state,city_a,city_b
1444c08e64d55fb3c25f0f09c07ffcf2,2b402d5dc42554061f8ea98d1916f148,BA,lauro de freitas,irece
1444c08e64d55fb3c25f0f09c07ffcf2,398cb257329ef7af7f1943a8974a3cbc,BA,lauro de freitas,salvador
1444c08e64d55fb3c25f0f09c07ffcf2,4221a7df464f1fe2955934e30ff3a5a1,BA,lauro de freitas,bahia
1444c08e64d55fb3c25f0f09c07ffcf2,43753b27d77860f1654aa72e251a7878,BA,lauro de freitas,barro alto
1444c08e64d55fb3c25f0f09c07ffcf2,4aba391bc3b88717ce08eb11e44937b2,BA,lauro de freitas,arraial d'ajuda (porto seguro)
1444c08e64d55fb3c25f0f09c07ffcf2,4fb41dff7c50136976d1a5cf004a42e2,BA,lauro de freitas,feira de santana
1444c08e64d55fb3c25f0f09c07ffcf2,651530bf5c607240ccdd89a30c9c9712,BA,lauro de freitas,ipira
1444c08e64d55fb3c25f0f09c07ffcf2,659e8466eb3ff1b0e8740d74fb7bbedd,BA,lauro de freitas,eunapolis
1444c08e64d55fb3c25f0f09c07ffcf2,75d34ebb1bd0bd7dde40dd507b8169c3,BA,lauro de freitas,salvador
1444c08e64d55fb3c25f0f09c07ffcf2,a3dd39f583bc80bd8c5901c95878921e,BA,lauro de freitas,salvador


The same self-join aggregated per state shows how many such pairs exist in total:

In [41]:
%%sql
SELECT
    s1.seller_state,
    (SELECT COUNT(*) FROM olist_sellers AS s
     WHERE s.seller_state = s1.seller_state)                        AS n_sellers,
    (SELECT COUNT(DISTINCT s.seller_city) FROM olist_sellers AS s
     WHERE s.seller_state = s1.seller_state)                        AS n_cities,
    COUNT(*)                                                        AS cross_city_pairs
FROM olist_sellers AS s1
JOIN olist_sellers AS s2
    ON s1.seller_state = s2.seller_state
   AND s1.seller_id < s2.seller_id
   AND s1.seller_city IS DISTINCT FROM s2.seller_city
GROUP BY s1.seller_state
ORDER BY cross_city_pairs DESC, s1.seller_state
LIMIT 8;

seller_state,n_sellers,n_cities,cross_city_pairs
SP,1849,261,1457751
PR,349,67,51665
MG,244,82,27184
SC,190,65,17208
RJ,171,38,10151
RS,129,51,7698
GO,40,12,519
ES,23,11,231


**Interpretation:** the first rows come from Bahia (BA), the first state in alphabetical order with more than one seller: seller `1444c08e…` from Lauro de Freitas pairs with every Bahia seller outside its city. The summary shows why such a list cannot be used as it is: São Paulo alone gives 1,457,751 cross-city pairs, because the number of pairs grows with the square of the number of sellers. A real regional comparison would aggregate per city or pick a few peers per seller.

The output also exposes a data-quality issue. `seller_city` is free text, so the list contains `bahia`, which is a state rather than a city, and both `porto seguro` and `arraial d'ajuda (porto seguro)`. São Paulo is spelled as `sao paulo`, `sp`, `sao paulo - sp`, `sao paulop` and more, so some "different city" pairs are actually the same city.

### 4.5 Anti-join: orders without a review

**Business-question:** Which orders have no review at all, and in which statuses do they occur?

The first query is the `LEFT JOIN ... IS NULL` form: an order that finds no review keeps `NULL` in every review column, including the primary key `review_row_id`, which is never `NULL` in a real row.

In [42]:
%%sql
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp
FROM olist_orders AS o
LEFT JOIN olist_order_reviews AS rv
    ON rv.order_id = o.order_id
WHERE rv.review_row_id IS NULL
ORDER BY o.order_purchase_timestamp, o.order_id
LIMIT 20;

order_id,customer_id,order_status,order_purchase_timestamp
cda873529ca7ab71f677d5ec11a40304,76c74aaff2f3f7355f46d9818ad092b8,shipped,2016-10-05 16:57:30
5da2004d85f2828349d887c989da2adc,95f592086150f0c95d3082aea047b816,delivered,2016-10-05 20:20:59
02190241f7190a1f3c7e0df95a749c6a,55c9044401ffab0c717f24405e2e042f,delivered,2016-10-07 10:52:22
0efd0bc268d34da3f01f4ff25a6d4335,95446917717bb58d553d107d0f1668f6,shipped,2016-10-07 15:53:31
c776863a93dc0740c6e7d78104b21413,1710b798ebdc5db6665c815b90a64999,delivered,2016-10-10 15:01:27
d730d8d0ea3149120703933fdf98b2d4,ce2a66464da8b80dedd7639a6e489a58,shipped,2017-01-13 20:12:41
05cd156f0f47579a36d7a52724fc31b8,976224a95faf328b17654b1b1411d5e1,delivered,2017-01-14 18:27:28
719b940ab2f1e351ffb14a0a02ffe3c4,ede9a1daf58d256c0d648ef7e1479010,delivered,2017-01-16 13:01:51
fbd82b1fe98967de0db29b6cf134d00a,45f5c9d984dbfa2105e3544cc85133bf,delivered,2017-01-18 08:55:36
171398a0cd843c65ac78aee9e05a1f49,f0244902c07b7c0799e468218579a925,delivered,2017-01-18 16:07:37


The same anti-join written with `NOT EXISTS`, counted per status:

In [43]:
%%sql
SELECT
    o.order_status,
    COUNT(*)                                                         AS orders_without_review,
    (SELECT COUNT(*) FROM olist_orders AS a
     WHERE a.order_status = o.order_status)                          AS orders_in_status,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM olist_orders AS a
                              WHERE a.order_status = o.order_status), 2) AS pct_without_review
FROM olist_orders AS o
WHERE NOT EXISTS (
    SELECT 1
    FROM olist_order_reviews AS rv
    WHERE rv.order_id = o.order_id
)
GROUP BY o.order_status
ORDER BY orders_without_review DESC;

order_status,orders_without_review,orders_in_status,pct_without_review
delivered,646,96478,0.67
shipped,75,1107,6.78
canceled,20,625,3.20
unavailable,14,609,2.30
processing,6,301,1.99
invoiced,5,314,1.59
created,2,5,40.00


**Interpretation:** 768 orders have no review. Most of them are delivered (646), but as a share delivered orders almost always get a review (only 0.67% are missing it), while 6.78% of shipped orders and 2 of the 5 stuck `created` orders have none. According to the dataset description, the satisfaction survey is sent when the order arrives or when the estimated delivery date is reached, so a missing review often goes together with an order that never arrived. The earliest unreviewed orders are from October 2016, the first months of the dataset. For a satisfaction model, "no review" is informative in itself and should not be filled with an average score.

### JOIN coverage

| Required JOIN | Query |
|---|---|
| `INNER JOIN` on at least three tables | 4.1 (order items, orders, products, sellers) |
| `LEFT JOIN` with `COALESCE` for features | 4.2 |
| `FULL OUTER JOIN` comparing two sets | 4.3 |
| `SELF JOIN` | 4.4 |
| anti-join | 4.5 (`LEFT JOIN ... IS NULL` and `NOT EXISTS`) |

## 5. `GROUP BY`, `HAVING` and aggregates

### 5.1 Revenue per product category

**Business-question:** Which product categories sold more than R$10,000 in items, and how large are their orders and prices?

`HAVING SUM(oi.price) > 10000` keeps only categories with meaningful sales; the long tail of small categories would only add noise to a category ranking.

In [44]:
%%sql
SELECT
    p.product_category_name,
    COUNT(DISTINCT oi.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS gross_revenue,
    SUM(oi.freight_value)::NUMERIC(12, 2) AS freight,
    AVG(oi.price)::NUMERIC(8, 2) AS avg_item_price
FROM olist_order_items AS oi
JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY p.product_category_name
HAVING SUM(oi.price) > 10000
ORDER BY gross_revenue DESC NULLS LAST;

product_category_name,n_orders,gross_revenue,freight,avg_item_price
beleza_saude,8836,1258681.34,182566.73,130.16
relogios_presentes,5624,1205005.68,100535.93,201.14
cama_mesa_banho,9417,1036988.68,204693.04,93.30
esporte_lazer,7720,988048.97,168607.51,114.34
informatica_acessorios,6689,911954.32,147318.08,116.51
moveis_decoracao,6449,729762.49,172749.30,87.56
cool_stuff,3632,635290.85,84039.10,167.36
utilidades_domesticas,5884,632248.66,146149.11,90.79
automotivo,3897,592720.11,92664.21,139.96
ferramentas_jardim,3518,485256.46,98962.75,111.63


**Interpretation:** 53 of the 74 categories (counting the missing category as one) pass the R$10,000 threshold, and together they make 99.4% of all item revenue, so `HAVING` removes a long tail without losing anything material. Health and beauty leads with R$1.26 million, ahead of watches and gifts (R$1.21 million) and bed, bath and table (R$1.04 million). The average item price shows two business models: `cama_mesa_banho` sells cheap items (R$93.30) in many orders, while `pcs` sells few but expensive ones (181 orders, R$1,098.34 per item). Products without a category (`None`) rank 21st with R$179,535, which is too much revenue to leave unclassified.

### 5.2 Average order value per customer state

**Business-question:** In which customer states is the average order value (AOV) highest, counting only states with at least 100 orders?

In [45]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    SUM(order_total)::NUMERIC(12, 2) AS gmv,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) >= 100
ORDER BY aov DESC NULLS LAST;

customer_state,n_orders,gmv,aov
PB,532,140987.81,265.01
AL,411,96229.40,234.13
RO,247,57558.02,233.03
PA,970,217647.11,224.38
TO,279,61354.42,219.91
PI,493,108132.28,219.34
SE,345,73032.32,211.69
RN,482,101895.08,211.40
CE,1327,275606.30,207.69
MT,903,186168.96,206.17


The states that `HAVING` removed, with the same metrics:

In [46]:
%%sql
WITH order_totals AS (
    SELECT
        o.order_id,
        c.customer_state,
        SUM(oi.price + oi.freight_value)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.order_id,
        c.customer_state
)
SELECT
    customer_state,
    COUNT(*) AS n_orders,
    (SUM(order_total) / NULLIF(COUNT(*), 0))::NUMERIC(8, 2) AS aov
FROM order_totals
GROUP BY customer_state
HAVING COUNT(*) < 100
ORDER BY n_orders;

customer_state,n_orders,aov
RR,46,218.80
AP,68,239.16
AC,81,242.84


**Interpretation:** AOV is highest in the north and northeast, in Paraíba (PB, R$265.01), Alagoas (AL, R$234.13) and Rondônia (RO, R$233.03), and lowest in São Paulo (R$143.12), which in turn brings by far the most orders (41,375) and GMV (R$5.92 million). Two effects add up. Buyers in remote states choose more expensive items (an average item price of R$191 in PB against R$110 in SP), and they pay more freight because most sellers are in the southeast: freight is 18.2% of the amount paid in PB and 12.1% in SP. `HAVING COUNT(*) >= 100` removed Roraima (46 orders), Amapá (68) and Acre (81). Their AOV of R$219–243 would put them near the top, but with fewer than 100 orders a few expensive purchases can move the average a lot.

**`WHERE` vs `HAVING` on this query.** `WHERE` filters rows before they are grouped, `HAVING` filters groups after the aggregates are computed. The condition "at least 100 orders" is about a group, because `COUNT(*)` only exists once the orders of a state are grouped together, so it can only go into `HAVING`; `WHERE COUNT(*) >= 100` is an error. A condition on single rows belongs in `WHERE`, for example `WHERE o.order_status = 'delivered'` inside `order_totals` would drop canceled orders before they reach any sum. Putting row conditions into `WHERE` also makes the query cheaper, because fewer rows are grouped.

### 5.3 Categories bought by repeat buyers (`STRING_AGG`)

**Business-question:** What do the biggest repeat buyers spend, and which categories do they buy?

The assignment's template uses `HAVING COUNT(DISTINCT o.order_id) >= 1`. After the `INNER JOIN` with orders every buyer has at least one order, so that condition removes nothing. Here it is raised to `>= 2`, so the query lists only buyers who came back, which is the group a retention analysis cares about.

In [47]:
%%sql
SELECT
    c.customer_unique_id,
    COUNT(DISTINCT o.order_id) AS n_orders,
    SUM(oi.price)::NUMERIC(12, 2) AS total_spend,
    STRING_AGG(
        DISTINCT COALESCE(p.product_category_name, '(unknown)'),
        ', ' ORDER BY COALESCE(p.product_category_name, '(unknown)')
    ) AS categories
FROM olist_customers AS c
JOIN olist_orders AS o
    ON o.customer_id = c.customer_id
JOIN olist_order_items AS oi
    ON oi.order_id = o.order_id
LEFT JOIN olist_products AS p
    ON p.product_id = oi.product_id
GROUP BY c.customer_unique_id
HAVING COUNT(DISTINCT o.order_id) >= 2
ORDER BY total_spend DESC NULLS LAST, c.customer_unique_id
LIMIT 20;

customer_unique_id,n_orders,total_spend,categories
da122df9eeddfedc1dc1f5349a1a690c,2,7388.00,eletroportateis
c8460e4251689ba205045f3ea17884a1,4,4080.00,telefonia
055ec572ac7f3c7bdd04a183830ebe59,2,3999.98,automotivo
59d66d72939bc9497e19d89c61a96d5f,2,3459.00,esporte_lazer
58c1b085b54c03a1f1ab5f13d64c2b1c,2,2999.98,pcs
d77aa95864ae5b42160937615628723a,2,2400.00,pcs
7b0eaf68a16e4808e5388c67345033c9,2,2238.42,bebes
86df00dc5fd68f4dd5d5945ca19f3ed6,3,2200.00,informatica_acessorios
6ddbc64bd04d40f7768ff088d94cbeb8,2,2108.00,eletrodomesticos
73601b1eec55943e90ce8d61253d5c09,2,2009.98,consoles_games


**Interpretation:** 2,913 buyers placed at least two orders with items, about 3% of all buyers. Even the top repeat buyers spend less than the top one-off buyers in 4.2: the leader `da122df9…` spent R$7,388 on small appliances (`eletroportateis`) over two orders. 15 of the top 20 stayed within a single category, and across all repeat buyers it is 1,275 of 2,913 (44%). Repeat purchases often serve the same need again, so "more from the same category" is a natural baseline for recommendations.

## 6. Set operations, window function and reflection

### 6.1 `UNION`: every state code with its role

**Business-question:** Which state codes occur among customers or sellers, and in which role?

`UNION` is chosen because it removes duplicates: each state should appear once per role. `UNION ALL` would keep one row per customer and per seller, about 102,000 rows, and would need a `DISTINCT` on top to say the same thing.

In [48]:
%%sql
SELECT customer_state AS state, 'customer' AS role
FROM olist_customers

UNION

SELECT seller_state AS state, 'seller' AS role
FROM olist_sellers
ORDER BY state, role;

state,role
AC,customer
AC,seller
AL,customer
AM,customer
AM,seller
AP,customer
BA,customer
BA,seller
CE,customer
CE,seller


**Interpretation:** 50 rows: 27 states with the role `customer` and 23 with the role `seller`. Every state code appears as a customer state, and the four states without a `seller` row (AL, AP, RR, TO) stand out directly in the sorted list.

### 6.2 `INTERSECT` and `EXCEPT`: states with both roles and states with customers only

**Business-question:** In which states are there both customers and sellers, and which states have customers but no seller?

`INTERSECT` returns exactly the states found in both sets; `EXCEPT` returns those found in the first set only. Both work on distinct values, so no `DISTINCT` is needed.

In [49]:
%%sql
SELECT customer_state AS state
FROM olist_customers

INTERSECT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;

state
AC
AM
BA
CE
DF
ES
GO
MA
MG
MS


In [50]:
%%sql
SELECT customer_state AS state
FROM olist_customers

EXCEPT

SELECT seller_state AS state
FROM olist_sellers
ORDER BY state;

state
AL
AP
RR
TO


**Interpretation:** `INTERSECT` returns 23 states, exactly the seller states, so every state with a seller also has customers. `EXCEPT` returns AL, AP, RR and TO, the same four `customers_only` states as the `FULL OUTER JOIN` in 4.3. The set operators answer the yes/no question in a few lines; the `FULL OUTER JOIN` is needed when the counts from both sides should appear in one row.

### 6.3 Window function: the latest order of every real buyer

**Business-question:** What is the most recent order of each real buyer?

`ROW_NUMBER()` numbers the orders of each `customer_unique_id` from newest to oldest; `order_id DESC` breaks ties between orders placed at the same second, so the numbering is deterministic. Keeping `rn = 1` gives one row per buyer, without collapsing the other columns as `GROUP BY` would.

In [51]:
%%sql
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        o.order_id,
        o.order_purchase_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    customer_unique_id,
    customer_id,
    order_id,
    order_purchase_timestamp
FROM ranked_orders
WHERE rn = 1
ORDER BY customer_unique_id
LIMIT 20;

customer_unique_id,customer_id,order_id,order_purchase_timestamp
0000366f3b9a7992bf8c76cfdf3221e2,fadbb3709178fc513abc1b2670aa1ad2,e22acc9c116caa3f2b7121bbb380d08e,2018-05-10 10:56:27
0000b849f77a49e4a4ce2b2a4ca5be3f,4cb282e167ae9234755102258dd52ee8,3594e05a005ac4d06a72673270ef9ec9,2018-05-07 11:11:27
0000f46a3911fa3c0805444483337064,9b3932a6253894a02c1df9d19004239f,b33ec3b699337181488304f362a6b734,2017-03-10 21:05:03
0000f6ccb0745a6a4b88665a16c9f078,914991f0c02ef0843c0e7010c819d642,41272756ecddd9a9ed0180413cc22fb6,2017-10-12 20:29:41
0004aac84e0df4da2b147fca70cf8255,47227568b10f5f58a524a75507e6992c,d957021f1127559cd947b62533f484f7,2017-11-14 19:45:42
0004bd2a26a76fe21f786e4fbd80607f,4a913a170c26e3c8052ed0202849b5a8,3e470077b690ea3e3d501cffb5e0c499,2018-04-05 19:33:16
00050ab1314c0e55a6ca13cf7181fecf,d2509c13692836fc0449e88cf9eb4858,d0028facea13f508e880202d7097a5a1,2018-04-20 12:57:23
00053a61a98854899e70ed204dd4bafe,a81ebb9b32f102298c0c89635b4b3154,44e608f2db00c74a1fe329de44416a4e,2018-02-28 11:15:41
0005e1862207bf6ccc02e4228effd9a0,3b37fb626fdf46cd99d37ec62afa88ff,ae76bef74b97bcb0b3e355e60d9a6f9c,2017-03-04 23:32:12
0005ef4cd20d2893f0d9fbd94d3c0d97,c59e8ff99836e90d8b457d4122dc34e9,01b330808c5819a6a3cb79b72f0b8288,2018-03-12 15:22:12


The same window over all buyers, compared with `olist_customer_dim` from 3.1, which picked the latest `customer_id` with `DISTINCT ON`. A second window, `COUNT(*) OVER (PARTITION BY buyer, timestamp)`, finds buyers whose latest orders were placed at the same second:

In [52]:
%%sql
WITH ranked_orders AS (
    SELECT
        c.customer_unique_id,
        o.customer_id,
        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY o.order_purchase_timestamp DESC, o.order_id DESC
        ) AS rn,
        COUNT(*) OVER (
            PARTITION BY c.customer_unique_id, o.order_purchase_timestamp
        ) AS orders_same_second
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
)
SELECT
    COUNT(*)                                                      AS buyers,
    COUNT(*) FILTER (WHERE r.orders_same_second > 1)              AS buyers_with_tied_latest_orders,
    COUNT(*) FILTER (WHERE r.customer_id <> d.latest_customer_id) AS different_from_dim
FROM ranked_orders AS r
JOIN olist_customer_dim AS d
    ON d.customer_unique_id = r.customer_unique_id
WHERE r.rn = 1;

buyers,buyers_with_tied_latest_orders,different_from_dim
96096,269,135


**Interpretation:** the query returns one row per real buyer; the first 20 are shown, and the comparison counts all 96,096. 269 buyers placed their latest orders at the very same second, and for 135 of them the two methods pick a different `customer_id`, because the tie-breakers differ: `order_id DESC` here, `customer_id` in 3.1. Both results are valid, but a pipeline should define one tie-breaking rule and reuse it everywhere, otherwise two tables built from the same data disagree about "the latest order".

### 6.4 Reflection

**Hardest JOIN.** `FULL OUTER JOIN`: either side can be `NULL`, so the key needs `COALESCE` and the label must test both sides. The riskiest joins, though, were the ordinary ones between items and reviews, because they silently change the grain.

**Hardest business question.** The self-join. "Pairs of sellers" sounds natural, but São Paulo alone gives 1.46 million pairs, so the honest question is about regional peers, which needs aggregation.

**Most useful feature for production ETL.** `ON CONFLICT`: it turns a load into an idempotent upsert, so a failed job can simply be rerun. `RETURNING` comes next as a free audit log.

**Window functions in ML.** Latest order per buyer, order sequence numbers, days since the previous order, rolling spend over recent orders and a seller's rank within its category, always computed only from data before the prediction date.

**Data quality.** 768 orders without a review and 547 with several, 789 shared `review_id` values, 610 products without a category, 58% empty comments, 775 orders without items, statuses that contradict delivery dates, free-text city names, and a data tail that pushes the reference date 47 days forward.

**`customer_unique_id`.** `customer_id` is issued per order, so frequency would be 1 for almost everyone and the 2,997 repeat buyers would be split into separate customers.

**Seller × order grain.** Items and reviews have different grains. Joining them directly counted 112,372 reviews instead of 99,247 and changed the rating of 1,264 sellers.

## 7. Final checks

In [53]:
def scalar(query: str):
    return q(query).iloc[0, 0]


assert scalar("SELECT current_setting('server_version_num')::INT") >= 160000

for raw_table in files:
    typed_table = raw_table.removesuffix("_raw")
    raw_n = scalar(f"SELECT COUNT(*) FROM {raw_table}")
    typed_n = scalar(f"SELECT COUNT(*) FROM {typed_table}")
    assert raw_n == typed_n, (typed_table, raw_n, typed_n)

alerts = q("SELECT alert_type, created_at FROM seller_alerts")
assert alerts["alert_type"].tolist() == ["late_delivery"]

assert scalar("SELECT COUNT(DISTINCT seller_id) FROM olist_order_items") == scalar("SELECT COUNT(*) FROM seller_score")
assert scalar("SELECT COUNT(*) FROM seller_score WHERE (tier IS NULL) <> (n_reviews = 0)") == 0

assert scalar("""
    SELECT COUNT(*) FROM olist_orders
    WHERE is_late IS DISTINCT FROM COALESCE(
        order_delivered_customer_date::DATE > order_estimated_delivery_date::DATE, FALSE)
""") == 0

n_buyers = scalar("SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers")
assert scalar("SELECT COUNT(*) FROM olist_customer_dim") == n_buyers
assert scalar("SELECT COUNT(*) FROM customer_segments") == n_buyers

segment_constraints = q("""
    SELECT contype, COUNT(*) AS n
    FROM pg_constraint
    WHERE conrelid = 'customer_segments'::regclass
    GROUP BY contype
""").set_index("contype")["n"]
assert segment_constraints.to_dict() == {"p": 1, "u": 1, "f": 1, "c": 5}

print(f"typed rows match raw rows for {len(files)} tables | buyers segmented: {n_buyers:,}")
print("All checks passed")

typed rows match raw rows for 6 tables | buyers segmented: 96,096
All checks passed
